# Attention U-Net + ConvNeXt — Contrail Segmentation

Run this notebook on **Kaggle** (free T4/P100 GPU, dataset already available).

**Preprocessing notes:**
- `mask_only=False` → 24-channel false-color input (3 channels × 8 time frames)
- No ImageNet normalization — backbone trained from scratch on this data
- Uses `convnext_base` backbone (Kaggle has 16 GB VRAM, use `convnext_tiny` if OOM)

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone repo & install dependencies

In [ ]:
import os
if not os.path.exists('/kaggle/working/contrail-segmentation'):
    !git clone https://github.com/aryangarg794/contrail-segmentation.git /kaggle/working/contrail-segmentation
%cd /kaggle/working/contrail-segmentation
!git checkout attention-unet-training
!git pull

In [ ]:
import subprocess, sys, re

result = subprocess.run([sys.executable, '-m', 'pip', 'show', 'torch'],
                        capture_output=True, text=True)
m = re.search(r'Version: (.+)', result.stdout)
current_ver = m.group(1).strip() if m else ''

if current_ver.startswith('2.4.0'):
    print(f'torch {current_ver} already installed — skipping restart')
else:
    print(f'torch {current_ver} detected — installing 2.4.0+cu121 (P100-compatible)...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--no-user', '--force-reinstall',
                    'torch==2.4.0', 'torchvision==0.19.0',
                    '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
    # Pin triton to version compatible with torch 2.4.0 (Kaggle ships triton 3.x which breaks it)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--no-user', '--force-reinstall', 'triton==2.3.1'], check=True)
    print('Done — restarting kernel...')
    import os
    os._exit(0)

In [ ]:
# Run this cell AFTER the kernel restarts (skip the cell above)
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'lightning', 'timm',
                'albumentations>=1.3,<2.0',
                'segmentation-models-pytorch',
                'hydra-core', 'wandb', 'dill', 'rich', 'tqdm', 'transformers'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

import importlib
sys.path.insert(0, '/kaggle/working/contrail-segmentation/src')
importlib.invalidate_caches()

import torch
print(f'torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
x = torch.tensor([1.0]).cuda()
print(f'CUDA test: PASSED {x}')

In [ ]:
import torch, torchvision, lightning
print('torch     :', torch.__version__)
print('torchvision:', torchvision.__version__)
print('lightning  :', lightning.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Compute capability:', torch.cuda.get_device_capability(0))

try:
    x = torch.tensor([1.0]).cuda()
    print('CUDA tensor test: PASSED', x)
except Exception as e:
    print('CUDA tensor test: FAILED —', e)

## 3. Dataset setup

Add the competition dataset via the Kaggle sidebar:  
`+ Add Data → google-research-identify-contrails-reduce-global-warming`  
Data will be at `/kaggle/input/google-research-identify-contrails-reduce-global-warming/`

In [ ]:
import os

DATA_ROOT = '/kaggle/input/competitions/google-research-identify-contrails-reduce-global-warming'
TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
META_PATH = os.path.join(DATA_ROOT, 'train_metadata.json')
print('Train dir exists:', os.path.exists(TRAIN_DIR))
print('Metadata exists: ', os.path.exists(META_PATH))

## 4. Patch data paths at runtime

In [ ]:
import os, pandas as pd
import contrail_segmentation.data.utils as data_utils

data_utils.DATA_DIR  = TRAIN_DIR
data_utils.META_PATH = META_PATH

metadata = pd.read_json(META_PATH)
available = set(os.listdir(TRAIN_DIR))
# record_id is float64 in JSON — must go through int() to avoid scientific notation in astype(str)
metadata = metadata[metadata['record_id'].apply(lambda x: str(int(x))).isin(available)].reset_index(drop=True)
data_utils.metadata = metadata

print(f'Dataset size: {len(data_utils.metadata)} records ({len(available)} on disk)')

## 5. W&B login

Add your W&B API key under *Kaggle Settings → Secrets* with the name `WANDB_API_KEY`.

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

wandb.login(key=UserSecretsClient().get_secret('wandb'))

## 6. Build transforms

No normalization — the backbone is trained from scratch on raw false-color values.

In [ ]:
import albumentations as A

train_transform = A.Compose([
    A.Affine(
        scale=(0.8, 1.2), rotate=(-180, 180), translate_percent=(-0.3, 0.3),
        border_mode=0, fill=0, p=0.5,
    ),
    A.HorizontalFlip(),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.3, p=0.5),
])

# val_transform is None — no augmentation, no normalization needed

## 7. Build dataloaders

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from contrail_segmentation.data.dataset import ContrailDataset

SEED        = 0
BATCH_SIZE  = 16
NUM_WORKERS = 2

torch.manual_seed(SEED)
np.random.seed(SEED)
generator = torch.Generator().manual_seed(SEED)

full_dataset = ContrailDataset(mask_only=False)

# Filter to only indices whose record actually exists on disk
available = set(os.listdir(TRAIN_DIR))
valid_indices = [
    i for i in range(len(full_dataset))
    if str(int(full_dataset.df_meta.loc[i]['record_id'])) in available
]
print(f'{len(valid_indices)} valid / {len(full_dataset)} total records')

indices = np.array(valid_indices)
np.random.shuffle(indices)
train_size   = int(0.8 * len(indices))
train_idx, val_idx = indices[:train_size], indices[train_size:]

train_set = ContrailDataset(mask_only=False, transform=train_transform)
val_set   = ContrailDataset(mask_only=False)

train_loader = DataLoader(Subset(train_set, train_idx), batch_size=BATCH_SIZE,
                          shuffle=True, generator=generator,
                          pin_memory=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(Subset(val_set, val_idx), batch_size=BATCH_SIZE,
                          pin_memory=True, num_workers=NUM_WORKERS)

print(f'Train: {len(train_idx)} | Val: {len(val_idx)}')

## 8. Build model

Using `convnext_base` — 16 GB VRAM on Kaggle T4/P100 can handle it.  
Switch to `convnext_tiny` if you hit OOM.

In [ ]:
from contrail_segmentation.models.attention_unet_convnext import AttentionUNetConvNeXt

BACKBONE = 'convnext_base'  # switch to 'convnext_tiny' if OOM
MAX_EPOCHS = 100
total_steps = MAX_EPOCHS * len(train_loader)

model = AttentionUNetConvNeXt(
    in_channels=24,
    backbone=BACKBONE,
    lr=1e-4,
    wd=1e-3,
    threshold=0.5,
    total_steps=total_steps,
)

total = sum(p.numel() for p in model.parameters())
print(f'Total params: {total:,} | Scheduler steps: {total_steps}')

## 9. Train

In [ ]:
from datetime import datetime
from lightning.pytorch import Trainer
from lightning.pytorch.loggers import WandbLogger

MAX_EPOCHS = 100
timestamp  = datetime.now().strftime('%d_%b_%Y__%Hh%Mm')
run_name   = f'attention_unet_{BACKBONE}_seed{SEED}_{timestamp}'

logger = WandbLogger(project='contrail-segmentation', name=run_name, save_dir='wandb_logs')

trainer = Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='gpu',
    devices=1,
    precision='16-mixed',
    log_every_n_steps=1,
    logger=logger,
)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

## 10. Find best threshold & test

In [ ]:
from contrail_segmentation.train.utils import find_best_threshold

best_thresh = find_best_threshold(model, val_loader)
model.threshold = best_thresh
model.mask_only = False
print(f'Best threshold: {best_thresh:.4f}')

test_metrics = trainer.test(model, dataloaders=val_loader)
print(test_metrics)

## 11. Save checkpoint

In [ ]:
torch.save(model.state_dict(), f'attention_unet_{BACKBONE}_{timestamp}.pt')
print('Saved.')